Load Data


In [1]:
import pandas as pd
import numpy as np

df_rc = pd.read_csv('../../../data/interim/renewal_calls_cleaned.csv')
print(f"Shape: {df_rc.shape}")
df_rc.head()

Shape: (153269, 28)


,call_id,call_direction,co_ref,call_date,membership_renewal_decision,serious_complaint,other_complaint,discussion_on_price_increase,renewal_impact_due_to_price_increase,discount_or_waiver_requested,...,competitor_benefits_mentioned,topic_introduced_by,percentage_price_increase_mentioned,monetary_price_increase_mentioned,price_range_mentioned,customer_asked_for_justification,customer_response,desire_to_cancel,discount_offered,call_number
0,5.950000e+11,OUT_BOUND,UB0899,2025-01-29,No,No,No,No,No,Yes,...,Discounts,Not Discussed,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,3
1,5.970000e+11,OUT_BOUND,HN5141,2025-02-26,No,No,No,No,No,No,...,Unknown,Not Discussed,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,2
2,5.950000e+11,OUT_BOUND,BP5009,2025-01-24,No,No,No,No,No,No,...,Better Service,Not Discussed,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1
3,6.520000e+11,OUT_BOUND,XP8119,2025-06-09,No,No,No,No,No,No,...,Unknown,Not Discussed,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1
4,5.370000e+11,OUT_BOUND,ZL7978,2024-08-20,No,No,No,No,No,No,...,Discounts,Not Discussed,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,28


Parse Date Column

In [2]:
df_rc['call_date'] = pd.to_datetime(df_rc['call_date'], errors='coerce')
print(f"Date range: {df_rc['call_date'].min()} to {df_rc['call_date'].max()}")

Date range: 2024-04-02 00:00:00 to 2026-02-13 00:00:00


Encode Call Direction

In [3]:
# IN_BOUND → 1, OUT_BOUND → 0
df_rc['is_inbound'] = (df_rc['call_direction'] == 'IN_BOUND').astype(int)

Encode Yes/No/Unknown Columns

In [4]:
yes_no_cols = [
    'serious_complaint',
    'other_complaint',
    'discussion_on_price_increase',
    'renewal_impact_due_to_price_increase',
    'discount_or_waiver_requested',
    'call_reschedule_request',
    'agent_flagged_membership_status_alert',
    'agent_renewal_initiation',
    'explicit_competitor_mention',
    'explicit_switching_intent',
    'mentioned_competitors',
    'price_switching_mentioned',
    'percentage_price_increase_mentioned',
    'monetary_price_increase_mentioned',
    'customer_asked_for_justification',
    'discount_offered',
]

for col in yes_no_cols:
    df_rc[col] = df_rc[col].map({
        'Yes'          : 1,
        'No'           : 0,
        'Not Discussed': 0,
        'Unknown'      : 0,
    })

Encode Remaining Categorical Columns

In [5]:
# membership_renewal_decision
df_rc['membership_renewal_decision'] = df_rc['membership_renewal_decision'].map({
    'Yes'    : 1,
    'No'     : 0,
    'Unknown': 0,
})

# desire_to_cancel — ordinal by risk
df_rc['desire_to_cancel_encoded'] = df_rc['desire_to_cancel'].map({
    'Renewed'           : 0,
    'Not Discussed'     : 0,
    'Unknown'           : 0,
    'Desired to Cancel' : 1,
})

# customer_response — ordinal
df_rc['customer_response_encoded'] = df_rc['customer_response'].map({
    'Positive'     :  1,
    'Neutral'      :  0,
    'Not Discussed':  0,
    'Unknown'      :  0,
    'Negative'     : -1,
})

# topic_introduced_by — who raised the renewal topic?
df_rc['topic_by_customer'] = (df_rc['topic_introduced_by'] == 'Customer').astype(int)
df_rc['topic_by_agent']    = (df_rc['topic_introduced_by'] == 'Agent').astype(int)

# competitor_value_comparison — is competitor seen as better?
df_rc['competitor_better_value'] = (df_rc['competitor_value_comparison'] == 'Better Value').astype(int)
df_rc['competitor_similar_value'] = (df_rc['competitor_value_comparison'] == 'Similar Value').astype(int)

# competitor_benefits_mentioned — what benefit was mentioned?
df_rc['competitor_better_service'] = (df_rc['competitor_benefits_mentioned'] == 'Better Service').astype(int)
df_rc['competitor_cheaper']        = (df_rc['competitor_benefits_mentioned'] == 'Discounts').astype(int)

Handle price_range_mentioned

In [6]:
# Convert numeric values, set text to NaN
df_rc['price_range_value'] = pd.to_numeric(df_rc['price_range_mentioned'], errors='coerce')

# Flag: was a specific price range discussed?
df_rc['price_range_discussed_flag'] = df_rc['price_range_value'].notna().astype(int)

Feature Creation

In [7]:
# High risk call — strong churn signals in one call
df_rc['high_risk_call'] = (
    (df_rc['desire_to_cancel_encoded']        == 1) |
    (df_rc['serious_complaint']               == 1) |
    (df_rc['explicit_switching_intent']       == 1) |
    (df_rc['explicit_competitor_mention']     == 1)
).astype(int)

# Competitor threat flag — competitor mentioned AND seen as better
df_rc['competitor_threat_flag'] = (
    (df_rc['mentioned_competitors']   == 1) &
    (df_rc['competitor_better_value'] == 1)
).astype(int)

# Price pressure flag — price increase discussed AND impacted renewal
df_rc['price_pressure_flag'] = (
    (df_rc['discussion_on_price_increase']        == 1) &
    (df_rc['renewal_impact_due_to_price_increase']== 1)
).astype(int)

# Complaint count per call
df_rc['complaint_count'] = (
    df_rc['serious_complaint'] +
    df_rc['other_complaint']
)

# Competitor signal count per call
df_rc['competitor_signal_count'] = (
    df_rc['explicit_competitor_mention'] +
    df_rc['mentioned_competitors'] +
    df_rc['price_switching_mentioned'] +
    df_rc['competitor_value_comparison'].apply(
        lambda x: 1 if x in ['Better Value', 'Similar Value'] else 0
        if isinstance(x, str) else int(x == 1)
    )
)

# Agent engagement flag — agent actively tried to retain
df_rc['agent_active_retention'] = (
    (df_rc['agent_renewal_initiation']             == 1) |
    (df_rc['discount_offered']                     == 1) |
    (df_rc['agent_flagged_membership_status_alert']== 1)
).astype(int)

Aggregate Per Customer

In [8]:
agg_dict = {
    # Call counts
    'call_id'                              : 'count',
    'is_inbound'                           : 'sum',
    'call_number'                          : 'max',

    # Renewal decision
    'membership_renewal_decision'          : 'max',
    'desire_to_cancel_encoded'             : 'max',
    'customer_response_encoded'            : 'mean',

    # Complaint flags
    'serious_complaint'                    : 'max',
    'other_complaint'                      : 'max',
    'complaint_count'                      : 'sum',

    # Price flags
    'discussion_on_price_increase'         : 'max',
    'renewal_impact_due_to_price_increase' : 'max',
    'discount_or_waiver_requested'         : 'max',
    'discount_offered'                     : 'max',
    'price_pressure_flag'                  : 'max',
    'price_range_discussed_flag'           : 'max',
    'price_range_value'                    : 'mean',
    'percentage_price_increase_mentioned'  : 'max',
    'monetary_price_increase_mentioned'    : 'max',
    'price_range_mentioned'                : 'max',

    # Competitor flags
    'explicit_competitor_mention'          : 'max',
    'explicit_switching_intent'            : 'max',
    'mentioned_competitors'                : 'max',
    'price_switching_mentioned'            : 'max',
    'competitor_value_comparison'          : 'max',
    'competitor_benefits_mentioned'        : 'max',
    'competitor_better_value'              : 'max',
    'competitor_similar_value'             : 'max',
    'competitor_better_service'            : 'max',
    'competitor_cheaper'                   : 'max',
    'competitor_signal_count'              : 'sum',
    'competitor_threat_flag'               : 'max',

    # Agent behaviour
    'agent_renewal_initiation'             : 'max',
    'agent_flagged_membership_status_alert': 'max',
    'agent_active_retention'               : 'max',
    'call_reschedule_request'              : 'max',

    # Topic
    'topic_by_customer'                    : 'max',
    'topic_by_agent'                       : 'max',
    'customer_asked_for_justification'     : 'max',

    # Engineered features
    'high_risk_call'                       : 'max',
    'agent_active_retention'               : 'max',
}

df_rc_agg = df_rc.groupby('co_ref').agg(agg_dict).reset_index()

# Rename count column
df_rc_agg.rename(columns={
    'call_id'    : 'total_renewal_calls',
    'call_number': 'max_call_number',
}, inplace=True)

# Outbound calls
df_rc_agg['outbound_calls'] = (
    df_rc_agg['total_renewal_calls'] - df_rc_agg['is_inbound']
)

print(f"Shape after aggregation: {df_rc_agg.shape}")
df_rc_agg.head()

Shape after aggregation: (36303, 41)


,co_ref,total_renewal_calls,is_inbound,max_call_number,membership_renewal_decision,desire_to_cancel_encoded,customer_response_encoded,serious_complaint,other_complaint,complaint_count,...,competitor_threat_flag,agent_renewal_initiation,agent_flagged_membership_status_alert,agent_active_retention,call_reschedule_request,topic_by_customer,topic_by_agent,customer_asked_for_justification,high_risk_call,outbound_calls
0,AA0584,2,0,2,0,1,-0.5,0,0,0,...,0,1,1,1,0,0,1,1,1,2
1,AA0641,3,0,3,0,0,0.0,0,0,0,...,0,1,0,1,0,0,0,0,0,3
2,AA0784,3,0,3,0,0,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
3,AA0794,7,0,7,0,0,0.0,0,0,0,...,0,1,0,1,0,0,0,0,0,7
4,AA0882,1,0,1,0,0,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


Drop Columns No Longer Needed

In [9]:
drop_cols = [
    'call_direction',            # replaced by is_inbound
    'desire_to_cancel',          # replaced by desire_to_cancel_encoded
    'customer_response',         # replaced by customer_response_encoded
    'topic_introduced_by',       # replaced by topic_by_customer / topic_by_agent
    'competitor_value_comparison',  # replaced by competitor_better_value / similar_value
    'competitor_benefits_mentioned',# replaced by competitor_better_service / cheaper
    'price_range_mentioned',     # replaced by price_range_value + flag
]

df_rc_agg.drop(columns=[c for c in drop_cols if c in df_rc_agg.columns], inplace=True)

Final Check

In [10]:

print("Nulls:")
print(df_rc_agg.isnull().sum()[df_rc_agg.isnull().sum() > 0])

print(f"\nShape        : {df_rc_agg.shape}")
print(f"Unique co_ref: {df_rc_agg['co_ref'].nunique()}")

df_rc_agg.head()

Nulls:
price_range_value    32759
dtype: int64

Shape        : (36303, 38)
Unique co_ref: 36303


,co_ref,total_renewal_calls,is_inbound,max_call_number,membership_renewal_decision,desire_to_cancel_encoded,customer_response_encoded,serious_complaint,other_complaint,complaint_count,...,competitor_threat_flag,agent_renewal_initiation,agent_flagged_membership_status_alert,agent_active_retention,call_reschedule_request,topic_by_customer,topic_by_agent,customer_asked_for_justification,high_risk_call,outbound_calls
0,AA0584,2,0,2,0,1,-0.5,0,0,0,...,0,1,1,1,0,0,1,1,1,2
1,AA0641,3,0,3,0,0,0.0,0,0,0,...,0,1,0,1,0,0,0,0,0,3
2,AA0784,3,0,3,0,0,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
3,AA0794,7,0,7,0,0,0.0,0,0,0,...,0,1,0,1,0,0,0,0,0,7
4,AA0882,1,0,1,0,0,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


Save

In [11]:
df_rc_agg.to_csv('../../../data/interim/renewal_calls_features.csv', index=False)
print("Saved → data/interim/renewal_calls_features.csv")

Saved → data/interim/renewal_calls_features.csv
